In [ ]:
# 필요 라이브러리 

In [ ]:
import pandas as pd
import numpy as np
import os
import random
import map_to_grid
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from datetime import datetime
from haversine import haversine
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers
from sklearn.metrics import mean_absolute_error
from datetime import timedelta,datetime

In [ ]:
original_ais0 = pd.read_csv('test_ais2020.csv')
original_ais1 = pd.read_csv('test_ais2021.csv')
original_ais2 = pd.read_csv('test_ais2022.csv')
original_ais3 = pd.read_csv("ais_new_2022.csv")
original_port = pd.read_csv('port_list.csv')
original_cargo = pd.read_csv('cargo_list.csv', encoding='CP949')

In [ ]:
#original_ais = pd.concat([original_ais1,original_ais1,original_ais2])
original_ais = pd.concat([original_ais2,original_ais3])

In [ ]:
original_ais = original_ais.dropna(subset=['mmsi','lon','lat','ais_cdatetime','destination','utctime','eta'])
original_ais = original_ais.reset_index(drop=True)

In [ ]:
original_ais = original_ais.reset_index(drop=True)

In [ ]:
original_ais

In [ ]:
#좌표 이상한것 제거
for i in range(len(original_ais)):
    if original_ais['lat'][i]< -90 or original_ais['lat'][i]> 90 :
        original_ais = original_ais.drop(i,axis=0)
        
original_ais = original_ais.reset_index(drop=True)

for i in range(len(original_ais)):
    if original_ais['lon'][i]< -180 or original_ais['lon'][i]> 180 :
        original_ais = original_ais.drop(i,axis=0)
        
original_ais = original_ais.reset_index(drop=True)

In [ ]:
#original_ais.to_csv("final_original_ais.csv")

In [ ]:
#original_ais = pd.read_csv("final_original_ais.csv")
original_ais

In [ ]:
original_ais_v = original_ais.values

In [ ]:
# 좌표 그룹 지정
grid_x = []
grid_y = []
for i in range(len(original_ais_v)):
    x,y = map_to_grid.mapToGrid(original_ais_v[i][4],original_ais_v[i][3])
    grid_x.append(x)
    grid_y.append(y)

In [ ]:
original_ais['grid_x'] = grid_x
original_ais['grid_y'] = grid_y

In [ ]:
original_ais['grid_x_group'] = pd.qcut(original_ais.grid_x, q=10, labels=['A','B','C','D','E','F','G','H','I','J']) #10가지
original_ais['grid_y_group'] = pd.qcut(original_ais.grid_y, q=10, labels=['A','B','C','D','E','F','G','H','I','J'])

In [ ]:
grid_group = []
for i in range(len(original_ais)):
    grid_group.append(str(original_ais['grid_x_group'][i] + original_ais['grid_y_group'][i]))
    
original_ais['grid_group'] = grid_group

In [ ]:
original_ais_v = original_ais.values

In [ ]:
#utctime(unixtime) -> datetime Stamp
ais_utctime = []
for i in range(len(original_ais_v)):
    ais_utctime.append(datetime.fromtimestamp(int(original_ais_v[i][8])))

In [ ]:
#utctime(timestamp) 시간 단위 수정
for i in range(len(ais_utctime)):
    if 0 <= ais_utctime[i].minute < 30:
        ais_utctime[i] = ais_utctime[i].replace(minute=0, second=0)
    else:
        ais_utctime[i] = ais_utctime[i].replace(minute=30, second=0)

In [ ]:
#ais_cdatetime -> str 변경
ais_cdate = []
for i in range(len(original_ais_v)):
    ais_cdate.append(datetime.strptime(original_ais_v[i][-8], '%Y-%m-%d %H:%M:%S'))

In [ ]:
#ais_cdatetime 시간 단위 수정
for i in range(len(ais_cdate)):
    if 0 <= ais_cdate[i].minute < 30:
        ais_cdate[i] = ais_cdate[i].replace(minute=0, second=0)
    else:
        ais_cdate[i] = ais_cdate[i].replace(minute=30, second=0)

In [ ]:
#eta(unixtime) -> datetime Stamp
date_eta = []
for i in range(len(original_ais_v)):
    date_eta.append(datetime.fromtimestamp(int(original_ais_v[i][14])))

In [ ]:
#eta 시간 단위 수정
for i in range(len(date_eta)):
    if 0 <= date_eta[i].minute < 30:
        date_eta[i] = date_eta[i].replace(minute=0, second=0)
    else:
        date_eta[i] = date_eta[i].replace(minute=30, second=0)

In [ ]:
original_ais['utctime'] = ais_utctime
original_ais['eta'] = date_eta
original_ais['ais_cdatetime'] = ais_cdate

In [ ]:
timediff = []
for i in range(len(original_ais_v)):
    timediff.append((date_eta[i]-ais_cdate[i]).days)

In [ ]:
original_ais['timediff'] = timediff

In [ ]:
original_ais

In [ ]:
original_ais['mmsi'].value_counts()

In [ ]:
counts = original_ais['mmsi'].value_counts()
counts = counts < 25
index1 = counts[counts==True].index

In [ ]:
original_ais = original_ais.set_index('mmsi',drop=False)
original_ais = original_ais.drop(index1)
original_ais = original_ais.reset_index(drop=True)

In [ ]:
original_ais['mmsi'].value_counts()

In [ ]:
original_ais[original_ais['mmsi']==367361210]

# 학습 데이터 생성 및 학습

In [ ]:
test = original_ais[['mmsi',"destination",'eta','utctime','lon','lat','shiptype','grid_group']]
test = test.set_index('mmsi',drop=False)

In [ ]:
original_ais.columns

In [ ]:
test

In [ ]:
#index1 = counts[counts==True].index

In [ ]:
#test = test.drop(index1)

In [ ]:
le = LabelEncoder()
le.fit(test['mmsi'])
label_mmsi = le.transform(test['mmsi'])

In [ ]:
le.inverse_transform(label_mmsi)

In [ ]:
le2 = LabelEncoder()
le2.fit(test['destination'])
label_destination= le2.transform(test['destination'])

In [ ]:
le3 = LabelEncoder()
le3.fit(test['grid_group'])
label_gridgroup= le3.transform(test['grid_group'])

In [ ]:
test['label_mmsi'] = label_mmsi
test['label_destination'] = label_destination
test['label_gridgroup'] = label_gridgroup

In [ ]:
port_info = original_port[['name',"country_code","port_code",'lon','lat']]
port_info

In [ ]:
test = test.reset_index(drop=True)

In [ ]:
test_values = test.values
port_values = port_info.values

In [ ]:
test

In [ ]:
test_index = []
port_index = []
for i in range(len(test)):
    print(i)
    start = (test_values[i][5],test_values[i][4])
    for j in range(len(port_info)):
        goal = (port_values[j][4],port_values[j][3])
        if haversine(start,goal, unit='km')<=5:
            test_index.append(i)
            port_index.append(port_values[j][0])

In [ ]:
len(test_index)

In [ ]:
test_index

In [ ]:
port_index

In [ ]:
len(port_index)

In [ ]:
test_addpred = test
test_addpred

In [ ]:
mmsi_test = test_addpred[test_addpred['mmsi']==366941850] 

In [ ]:
mmsi_test['destination']

In [ ]:
mmsi_test['pred_ataport']

In [ ]:
test_addpred['mmsi'].value_counts()

In [ ]:
test_addpred['pred_ataport'] = np.nan
for i in range(len(test_index)):
    test_addpred['pred_ataport'][test_index[i]] = port_index[i]

In [ ]:
test_addpred

In [ ]:
test_addpred = test_addpred.dropna(subset=['pred_ataport'])
test_addpred

In [ ]:
le4 = LabelEncoder()
le4.fit(test_addpred['pred_ataport'])
label_preddestination= le4.transform(test_addpred['pred_ataport'])
test_addpred['label_pred_ataport'] = label_preddestination

In [ ]:
counts = test_addpred['mmsi'].value_counts()
counts = counts < 25
index1 = counts[counts==True].index

In [ ]:
test_addpred = test_addpred.set_index('mmsi',drop=False)
test_addpred

In [ ]:
#test_addpred = test_addpred.drop(index1)
test_addpred = test_addpred.reset_index(drop=True)

In [ ]:
le5 = LabelEncoder()
le5.fit(test_addpred['name'])
label_name= le5.transform(test_addpred['name'])
test_addpred['label_name'] = label_name

In [ ]:
test_addpred

## 목적지 예측

In [ ]:
test_addpred.columns

In [ ]:
csv_destpred = test_addpred[['mmsi','destination','eta','utctime','pred_ataport','shiptype','lat','lon']]

In [ ]:
csv_destpred.to_csv("dest_pred.csv")

In [ ]:
csv_timepred = test_addpred[['mmsi','destination','eta','utctime','shiptype','pred_ataport','lat','lon','timediff','ais_cdatetime']]

In [ ]:
csv_timepred

In [ ]:
pred_ata = []
for i in range(len(csv_timepred)):
    pred_ata.append(csv_timepred['ais_cdatetime'][i] + timedelta(days=int(csv_timepred['timediff'][i])))

In [ ]:
csv_timepred['pred_ata'] = pred_ata

In [ ]:
csv_timepred = csv_timepred[['mmsi','destination','eta','utctime','shiptype','pred_ataport','pred_ata','lat','lon']]
csv_timepred

In [ ]:
csv_timepred.to_csv("time_pred.csv")

In [ ]:
x = test_addpred[['label_mmsi','shiptype','label_gridgroup','dbow','label_name','rot','navi','label_name','lon','lat',
                  'draught','hdg','sog','cog','timediff']]
y = test_addpred['label_pred_ataport']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3)

In [ ]:
forest = RandomForestClassifier(n_estimators=10, max_depth=20, random_state=0)
forest.fit(x_train, y_train)

In [ ]:
y_pred = forest.predict(x_test)
print('정확도 :', metrics.accuracy_score(y_test, y_pred))

## 도착시간 예측

In [ ]:
test_timepred = test_addpred

In [ ]:
a = test_timepred['timediff'] > 60
b= test_timepred['timediff']< 120

In [ ]:
test_timepred = test_timepred[a]
test_timepred

In [ ]:
test_timepred.reset_index(drop=True,inplace=True)
test_timepred

In [ ]:
counts = test_timepred['mmsi'].value_counts()
counts = counts < 10
index1 = counts[counts==True].index

In [ ]:
test_timepred = test_timepred.set_index('mmsi',drop=False)
#test_timepred = test_timepred.drop(index1)
test_timepred = test_timepred.reset_index(drop=True)

In [ ]:
test_timepred

In [ ]:
test_timepred.iloc

In [ ]:
test_timepred_v = test_timepred.values

In [ ]:
test_timepred['timediff'][0]

In [ ]:
new_timediff = []
for i in range(len(test_timepred)):
    if 100 < test_timepred_v[i][32] <105:
        test_timepred['timediff'][i] =100
    elif 105 < test_timepred_v[i][32] <110:
        test_timepred['timediff'][i] =105

In [ ]:
test_timepred.iloc[1]

In [ ]:
test_timepred.to_csv

In [ ]:
test_timepred.columns

In [ ]:
x = test_timepred[['label_mmsi','shiptype','label_gridgroup','lon','lat','dbow','rot','navi','label_destination',
                   'hdg','sog','cog','label_pred_ataport']]
y = test_timepred['timediff']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3)
forest = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=0)
forest.fit(x_train, y_train)

In [ ]:
y_pred = forest.predict(x_test)
print('정확도 :', metrics.accuracy_score(y_test, y_pred))

In [ ]:
timedelta

In [ ]:
datetime.now()

In [ ]:
test_timepred.iloc[0]

In [ ]:
pred_time = []
for i in range(len(test_timepred)):
    pred_time.append(test_timepred['ais_cdatetime'][i] + timedelta(days=int(test_timepred['timediff'][i])))

In [ ]:
test_timepred['pred_time'] = pred_time

In [ ]:
test_timepred['ais_udatetime']

In [ ]:
test_timepred.to_csv("total_pred_data_2022.csv")

In [ ]:
x = test_timepred[['label_mmsi','shiptype','label_gridgroup','lon','lat','label_pred_ataport','hdg','sog','cog']]
y = test_timepred['timediff']

In [ ]:
y

In [ ]:
# 피처 스케일링
scaler = MinMaxScaler()
x_data_scaled = scaler.fit_transform(x)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_data_scaled, y, test_size=0.2, shuffle=True, random_state = 777)

In [ ]:
adam = keras.optimizers.Adam()

In [ ]:
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=x_train.shape[1]))
model.add(tf.keras.layers.Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(tf.keras.layers.Dropout(0.2))
model.add(Dense(1, activation='linear'))
model.compile(loss="mse", optimizer= adam)
model.summary()

In [ ]:
model.fit(x_train, y_train, epochs=100, batch_size=16, verbose=2)

In [ ]:
predict = model.predict(x_test)
mae = mean_absolute_error(y_test, predict)

In [ ]:
mae